# Silver Layer — CRM: Sales Details

Transforms the bronze `workspace.bronze.sales_details` table into `workspace.silver.sales_details` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `sls_ord_num` → `order_number`, `sls_cust_id` → `customer_id`) |
| 2 | **Date Conversion** | Converted `order_date`, `ship_date`, `due_date` from integer (YYYYMMDD format) to actual DATE type. `order_date` with value `0` (17 rows) set to NULL |
| 3 | **NULL Sales Handling** | Replaced NULL `sales_amount` values with `0` |
| 4 | **NULL Price Handling** | Replaced NULL `price` values with `0` |
| 5 | **Negative Sales Handling** | Replaced negative `sales_amount` values (min was -54) with `0` |
| 6 | **Negative Price Handling** | Replaced negative `price` values (min was -1701) with `0` |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `sls_ord_num` | `order_number` | Renamed |
| `sls_prd_key` | `product_key` | Renamed |
| `sls_cust_id` | `customer_id` | Renamed |
| `sls_order_dt` | `order_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_ship_dt` | `ship_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_due_dt` | `due_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_sales` | `sales_amount` | Renamed + NULLs → 0 + Negatives → 0 |
| `sls_quantity` | `quantity` | Renamed |
| `sls_price` | `price` | Renamed + NULLs → 0 + Negatives → 0 |

---

## Implementation

All transformations are implemented in **Spark SQL**:
- `TO_DATE()` — convert integer dates (YYYYMMDD) to actual DATE type
- `CASE WHEN` + `LENGTH()` — guard the order_date conversion (values shorter than 8 digits become NULL)
- `COALESCE()` — replace NULL values with 0
- `GREATEST()` — replace negative values with 0
- `CREATE OR REPLACE TABLE ... AS SELECT` — write to silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.sales_details AS
SELECT
    sls_ord_num                         AS order_number,
    sls_prd_key                         AS product_key,
    sls_cust_id                         AS customer_id,
    CASE
        WHEN LENGTH(CAST(sls_order_dt AS STRING)) = 8
        THEN TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
    END                                 AS order_date,
    TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') AS ship_date,
    TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') AS due_date,
    GREATEST(COALESCE(sls_sales, 0), 0) AS sales_amount,
    sls_quantity                        AS quantity,
    GREATEST(COALESCE(sls_price, 0), 0) AS price
FROM workspace.bronze.sales_details

In [0]:
%sql
SELECT * FROM workspace.silver.sales_details LIMIT 10